<a href="https://colab.research.google.com/github/Tharun-2006161/ai-mentor-portfolio1/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install -q google-genai pydantic

import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [25]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [26]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

        except ValidationError as e:
            if attempt == max_retries:
                raise

In [27]:
resume_text = """
Ravi Kumar
ravi@gmail.com
9876543210

B.Tech CSE
Aditya University
2025

Skills:
Python, Java, SQL

Projects:
Student Management System

Experience:
1 year internship
"""

parsed = extract_resume(resume_text)

print(parsed.model_dump_json(indent=2))

{
  "name": "Ravi Kumar",
  "email": "ravi@gmail.com",
  "phone": "9876543210",
  "education": [
    {
      "degree": "B.Tech CSE",
      "institution": "Aditya University",
      "year": 2025
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL"
  ],
  "projects": [
    "Student Management System"
  ],
  "experience_years": 1.0
}


In [28]:
try:
    bad = extract_resume('')
    print(bad)
except Exception as e:
    print("Caught gracefully:", type(e).__name__)

name='Jane Doe' email='jane.doe@example.com' phone='123-456-7890' education=[Education(degree='Master of Science in Computer Science', institution='University of Tech', year=2020), Education(degree='Bachelor of Science in Software Engineering', institution='State University', year=2018)] skills=['Python', 'Java', 'JavaScript', 'React', 'SQL', 'AWS', 'Machine Learning'] projects=['E-commerce Platform Development', 'AI-powered Recommendation System', 'Mobile App for Task Management'] experience_years=5.5
